In [1]:
import numpy as np
import pandas as pd

In [2]:
flow_data = np.load("data/flow_data_filled.npy")   
scenic_names = np.load("data/scenic_names_cleaned.npy")
embedding_code_chat = np.load("data/event_embedding.npy")  


In [3]:
print(flow_data.shape)
print(scenic_names.shape)
print(embedding_code_chat.shape)


(79, 365, 36)
(79,)
(365, 4096)


In [4]:
train_flow = flow_data[:, :-61, :]
train_llm = embedding_code_chat[:-61, :]
test_flow = flow_data[:, -61:, :]
test_llm = embedding_code_chat[-61:, :]


In [5]:
print(train_llm.shape)
print(test_llm.shape)


(304, 4096)
(61, 4096)


In [6]:
train_flow = train_flow.reshape(79, 304 * 36)

test_flow = test_flow.reshape(79, 61 * 36)


In [7]:
print(train_flow.shape)
print(test_flow.shape)

(79, 10944)
(79, 2196)


In [8]:
train_llm_expanded = np.repeat(train_llm, repeats=36, axis=0) 
test_llm_expanded = np.repeat(test_llm, repeats=36, axis=0)  
print(train_llm_expanded.shape) 
print(test_llm_expanded.shape)  


(10944, 4096)
(2196, 4096)


In [9]:
x_train= []
y_train= []
x_train_llm = []
y_train_llm = []
for kk in range(24,10936):
    x_train = x_train + [np.array(train_flow[:,kk-24:kk])]
    y_train = y_train + [np.array(train_flow[:,kk:kk+8])]
    x_train_llm.append(train_llm_expanded[kk]) 
    y_train_llm.append(train_llm_expanded[kk])   

 
x_train = np.array(x_train).astype(float)
y_train = np.array(y_train).astype(float)
x_train_llm = np.array(x_train_llm).astype(float)  
y_train_llm = np.array(y_train_llm).astype(float)  


In [10]:
print(x_train.shape, y_train.shape)
print(x_train_llm.shape, y_train_llm.shape)


(10912, 79, 24) (10912, 79, 8)
(10912, 4096) (10912, 4096)


In [11]:
from sklearn.model_selection import train_test_split

x_tra, x_val, y_tra, y_val  = train_test_split(
    x_train[:,:,:,np.newaxis], y_train, test_size=1/8, shuffle=True, random_state=42 
)

print(f"x_train shape: {x_tra.shape}, x_val shape: {x_val.shape}")
print(f"y_train shape: {y_tra.shape}, y_val shape: {y_val.shape}")


x_train shape: (9548, 79, 24, 1), x_val shape: (1364, 79, 24, 1)
y_train shape: (9548, 79, 8), y_val shape: (1364, 79, 8)


In [12]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(x_train))
train_idx, val_idx = train_test_split(
    indices, test_size=1/8, shuffle=True, random_state=42
)

x_tra = x_train[train_idx][:, :, :, np.newaxis]
x_val = x_train[val_idx][:, :, :, np.newaxis]
y_tra = y_train[train_idx]
y_val = y_train[val_idx]

event_tra = x_train_llm[train_idx]
event_val = x_train_llm[val_idx]

In [13]:
print(f"x_train shape: {x_tra.shape}, x_val shape: {x_val.shape}")
print(f"y_train shape: {y_tra.shape}, y_val shape: {y_val.shape}")
print(f"event_tra shape: {event_tra.shape}, event_val shape: {event_val.shape}")

x_train shape: (9548, 79, 24, 1), x_val shape: (1364, 79, 24, 1)
y_train shape: (9548, 79, 8), y_val shape: (1364, 79, 8)
event_tra shape: (9548, 4096), event_val shape: (1364, 4096)


In [14]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import keras.backend as K
import numpy as np


def rmse(y_true, y_pred):

    return K.sqrt(K.mean(K.square(y_pred - y_true)))


adj = np.load("data/gaussian_adjacency.npy")

def normalize_adj(adj):
    adj += np.eye(adj.shape[0])
    degree = np.sum(adj, axis=1)
    degree_mat = np.diag(np.power(degree, -0.5))
    return degree_mat @ adj @ degree_mat

norm_adj = normalize_adj(adj)


class DynamicAdjGenerator(layers.Layer):
    def __init__(self, static_adj, num_nodes, window_size=6):
        super().__init__()
        self.static_adj = tf.constant(static_adj, dtype=tf.float32) 
        self.num_nodes = num_nodes
        self.window = window_size
        
        self.temporal_net = tf.keras.Sequential([
            layers.Conv1D(12, 3, activation='elu',padding="same"),   
            layers.Conv1D(2, 3, activation='elu',padding="same"),
            layers.Dropout(0.05), 
            layers.Flatten(),
            layers.LayerNormalization(),
            layers.Dense(12, activation='elu'), 
        ])
        self.dense_1 = layers.Dense(num_nodes, activation='elu')
        self.dense_2 = layers.Dense(num_nodes, activation='elu')
        
    def call(self, x):
        window_data = x[:, -self.window:] 
        temporal_weights = self.temporal_net(tf.transpose(window_data, [0,2,1,3])) 
        denweight_1 = tf.expand_dims(self.dense_1(temporal_weights),axis=1)
        denweight_2 = tf.expand_dims(self.dense_2(temporal_weights),axis=2)       
        temporal_weights = denweight_1 + denweight_2
        temporal_weights = tf.reshape(temporal_weights, [-1, self.num_nodes, self.num_nodes])
 
        dyn_adj = self.static_adj * tf.nn.sigmoid(temporal_weights) 
        return dyn_adj


class DynamicGCNLayer(layers.Layer):
    def __init__(self, output_dim, activation=None):
        super().__init__()
        self.output_dim = output_dim
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1], self.output_dim),
                                 initializer='glorot_uniform',
                                 name='gcn_weights')
        super().build(input_shape)

    def call(self, inputs, adjacency):
        adj = tf.repeat(adjacency, tf.shape(inputs)[0] // tf.shape(adjacency)[0], axis=0)

        h = tf.matmul(inputs, self.w)
        output = tf.einsum('bij,bjk->bik', adj, h)
        return self.activation(output) if self.activation else output

def spatio_temporal_dual_attention_encoder(
    inputs,
    head_size,
    num_heads,
    ff_dim,
    dropout=0.0
):

    B = tf.shape(inputs)[0]
    T = inputs.shape[1]
    D = inputs.shape[2]
    d_model = D
    
    temporal_encoding = tf.Variable(
        initial_value=tf.random.normal([1, T, D]),
        trainable=True,
        name="temporal_positional_encoding"
    )

    x_temporal = inputs + temporal_encoding

    spatial_encoding = tf.Variable(
        initial_value=tf.random.normal([1, D, T]),
        trainable=True,
        name="spatial_node_encoding"
    )


    x_temporal = layers.LayerNormalization(epsilon=1e-6)(x_temporal)
    temporal_attn = layers.MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(x_temporal, x_temporal)  

    temporal_attn = layers.Dropout(dropout)(temporal_attn)
    temporal_attn = layers.Conv1D(filters=d_model, kernel_size=1)(temporal_attn)
    
    x_spatial = layers.Permute((2, 1))(inputs)
    x_spatial = x_spatial + spatial_encoding
    x_spatial = layers.LayerNormalization(epsilon=1e-6)(x_spatial)

    spatial_attn = layers.MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(x_spatial, x_spatial)   

    spatial_attn = layers.Dropout(dropout)(spatial_attn)
    

    spatial_attn = layers.Permute((2, 1))(spatial_attn)
    spatial_attn = layers.Conv1D(filters=d_model, kernel_size=1)(spatial_attn)
    attn_out = layers.Add()([temporal_attn, spatial_attn])
    res = layers.Add()([inputs, attn_out])
    x = layers.LayerNormalization(epsilon=1e-6)(res)

    x = layers.Conv1D(
        filters=ff_dim,
        kernel_size=1,
        activation="relu"
    )(x)

    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(
        filters=d_model,
        kernel_size=1
    )(x)
    outputs = layers.Add()([res, x])

    return outputs

In [15]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import keras.backend as K

from tensorflow.keras import backend as K

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def huber_loss(y_true, y_pred, delta=1.0):
    error = y_pred - y_true  
    abs_error = K.abs(error) 
    
    condition = K.less(abs_error, delta)
    squared_loss = 0.5 * K.square(error) 
    linear_loss = delta * abs_error - 0.5 * K.square(delta) 
    
    loss = K.switch(condition, 
                   squared_loss, 
                   linear_loss)
    
    return K.mean(loss)

num_nodes   = 79
in_steps    = 24
out_steps   = 8
llm_dim     = 4096


head_size=128
num_heads=8
ff_dim=8
dropout=0.2


adj = np.load("data/semantic_adjacency.npy")
norm_adj_llm = normalize_adj(adj)




flow_input = keras.Input(shape=(num_nodes, in_steps), name="flow_input")  
llm_input  = keras.Input(shape=(llm_dim,), name="llm_input")              

gate_hidden = layers.Dense(512, activation="relu")(llm_input)             
llm_gate    = layers.Dense(num_nodes * in_steps, activation="sigmoid")(gate_hidden)  
llm_gate    = layers.Reshape((num_nodes, in_steps), name="llm_gate")(llm_gate)       


modulated = layers.Multiply(name="feature_gated")([flow_input, llm_gate]) 

seasonal_output = modulated
adj_gen1 = DynamicAdjGenerator(norm_adj, num_nodes)
dyn_adj1 = adj_gen1(tf.reshape(seasonal_output,[-1, in_steps, 79, 1])) 

adj_gen2 = DynamicAdjGenerator(norm_adj_llm, num_nodes)
dyn_adj2 = adj_gen2(tf.reshape(seasonal_output,[-1, in_steps, 79, 1]))  


dyn_adj = dyn_adj1 + dyn_adj2 

seasonal_output = tf.reshape(seasonal_output,[-1, 79, 1]) 
seasonal_output = DynamicGCNLayer(12, 'elu')(seasonal_output, dyn_adj)
seasonal_output = layers.Dense(1)(seasonal_output)
seasonal_output = tf.reshape(seasonal_output,[-1, in_steps, 79])  

x = seasonal_output  


x = spatio_temporal_dual_attention_encoder(x, head_size, num_heads, ff_dim, dropout)
x = spatio_temporal_dual_attention_encoder(x, head_size, num_heads, ff_dim, dropout)
x = spatio_temporal_dual_attention_encoder(x, head_size, num_heads, ff_dim, dropout)

z = layers.GlobalAveragePooling1D(name="gap_time")(x)       


dense_list = [layers.Dense(num_nodes, activation="elu") for _ in range(out_steps)]
outs = [dense(z) for dense in dense_list]                  
y = tf.stack(outs, axis=1)                                 

final_output = tf.transpose(y, perm=[0, 2, 1], name="y_pred")  


model = Model([flow_input, llm_input], final_output, name="Flow_LLMS_GatedTransformer")
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=huber_loss, metrics=["mae"])


model.summary()


Model: "Flow_LLMS_GatedTransformer"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 llm_input (InputLayer)         [(None, 4096)]       0           []                               
                                                                                                  
 dense (Dense)                  (None, 512)          2097664     ['llm_input[0][0]']              
                                                                                                  
 dense_1 (Dense)                (None, 1896)         972648      ['dense[0][0]']                  
                                                                                                  
 flow_input (InputLayer)        [(None, 79, 24)]     0           []                               
                                                                         

In [16]:
print(x_tra.shape,x_val.shape)
print(event_tra.shape,event_val.shape)
print(y_tra.shape,y_val.shape)
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', 
                               patience=50,       
                               restore_best_weights=True)  

model.fit([x_tra, event_tra], y_tra ,
              validation_data=([x_val, event_val], y_val ),
              epochs=1500, batch_size=24, callbacks=[early_stopping])


(9548, 79, 24, 1) (1364, 79, 24, 1)
(9548, 4096) (1364, 4096)
(9548, 79, 8) (1364, 79, 8)
Epoch 1/1500
398/398 [==============================] - 15s 23ms/step - loss: 2284.0208 - mae: 2284.5195 - val_loss: 1908.6736 - val_mae: 1909.1742
Epoch 2/1500
398/398 [==============================] - 8s 21ms/step - loss: 1635.4968 - mae: 1635.9968 - val_loss: 1436.8630 - val_mae: 1437.3628
Epoch 3/1500
398/398 [==============================] - 8s 21ms/step - loss: 1262.1107 - mae: 1262.6105 - val_loss: 1227.3132 - val_mae: 1227.8130
Epoch 4/1500
398/398 [==============================] - 8s 21ms/step - loss: 1146.4664 - mae: 1146.9664 - val_loss: 1109.6305 - val_mae: 1110.1302
Epoch 5/1500
398/398 [==============================] - 8s 21ms/step - loss: 1044.3937 - mae: 1044.8940 - val_loss: 1047.9056 - val_mae: 1048.4056
Epoch 6/1500
398/398 [==============================] - 8s 21ms/step - loss: 955.2278 - mae: 955.7277 - val_loss: 930.1265 - val_mae: 930.6261
Epoch 7/1500
398/398 [=========

In [17]:
import numpy as np

x_tst = np.load("data/test_data_npy/x_test.npy")
y_tst = np.load("data/test_data_npy/y_test.npy")
event_tst = np.load("data/test_data_npy/event_test.npy")

print("x_test:", x_tst.shape)
print("y_test:", y_tst.shape)
print("event_test:", event_tst.shape)


x_test: (2164, 79, 24, 1)
y_test: (2164, 79, 8)
event_test: (2164, 4096)


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import math

def smape(y_true, y_pred):
    return np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100

def wmape(y_true, y_pred):
    weights = y_true / np.sum(y_true)
    absolute_errors = np.abs((y_true - y_pred) / (y_true+ (y_true==0)*0.00001))
    return np.sum(weights * absolute_errors) * 100

def fenduan(y_test, y_predict):
    MAE = mean_absolute_error(y_test, y_predict)
    RMSE = math.sqrt(mean_squared_error(y_test, y_predict))
    WMAPE = wmape(y_test, y_predict)

    print("mean_absolute_error:", MAE)
    print("mean_squared_error:", mean_squared_error(y_test, y_predict))
    print("rmse:", RMSE)
    print("WMAPE:", WMAPE)

    print("MAE:", round(MAE, 2), "RMSE:", round(RMSE, 4), "WMAPE:", round(WMAPE, 2))

In [19]:
y_pred = model.predict([x_tst, event_tst])
fenduan(y_tst.reshape([-1]),y_pred.reshape([-1]))

68/68 [==============================] - 1s 6ms/step
mean_absolute_error: 518.1272540211454
mean_squared_error: 2764269.2340638787
rmse: 1662.6091645554823
WMAPE: 16.19487515316383
MAE: 518.13 RMSE: 1662.6092 WMAPE: 16.19


In [20]:
import numpy as np
y_pred = y_pred[:,:,:-4]
y_tst = y_tst[:,:,:-4]
print(y_pred.shape)
print(y_tst.shape)

(2164, 79, 4)
(2164, 79, 4)


In [21]:
fenduan(y_tst.reshape([-1]),y_pred.reshape([-1]))

mean_absolute_error: 412.38160838340474
mean_squared_error: 1521978.9851917722
rmse: 1233.6851240052188
WMAPE: 12.885589068534559
MAE: 412.38 RMSE: 1233.6851 WMAPE: 12.89
